# M8 · Calibration & class imbalance

_Curriculum · Domain 1 · Ranking & Recommenders_

**Make predicted probabilities mean what downstream systems think they mean.**

We compute ECE, fit Platt scaling, and draw a reliability diagram for rare click-style labels. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
# Setup - CPU-only and deterministic.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

rng = np.random.default_rng(8)

## First, look at raw scores

A model can be good at ranking but overconfident as a probability source. Calibration asks whether $\Pr(Y=1\mid \hat p=p)=p$.

In [ ]:
n = 3000
z = rng.normal(size=n)
raw_p = 1.0 / (1.0 + np.exp(-(1.4 * z - 2.6)))
true_p = 0.65 * raw_p
y = (rng.random(n) < true_p).astype(int)

print("positive rate:", round(y.mean(), 4))
print("mean raw probability:", round(raw_p.mean(), 4))

## The calibration metric

Expected calibration error bins predictions and computes

$$ECE=\sum_b \frac{n_b}{n}|acc(b)-conf(b)|$$

where `acc` is observed frequency and `conf` is average predicted probability.

### Step 1 - Compute reliability bins

We bin by predicted probability, then compare mean prediction with observed click rate.

In [ ]:
bins = np.linspace(0.0, 1.0, 8)
bin_id = np.digitize(raw_p, bins) - 1
rows = []
for b in range(len(bins) - 1):
    mask = bin_id == b
    if mask.sum() > 0:
        rows.append((b, mask.sum(), raw_p[mask].mean(), y[mask].mean()))

reliability = pd.DataFrame(rows, columns=["bin", "n", "conf", "acc"])
reliability["gap"] = (reliability["acc"] - reliability["conf"]).abs()
ece_raw = ((reliability["n"] / n) * reliability["gap"]).sum()

print(reliability.round(4))
print("raw ECE:", round(ece_raw, 4))

assert ece_raw > 0.01

### Step 2 - Fit Platt scaling

Platt scaling learns $\sigma(a z+b)$ on held-out labels. Here we use the raw logit-like score `z`.

In [ ]:
cal = LogisticRegression()
cal.fit(z.reshape(-1, 1), y)
cal_p = cal.predict_proba(z.reshape(-1, 1))[:, 1]

print("raw log loss:", round(log_loss(y, raw_p), 4))
print("cal log loss:", round(log_loss(y, cal_p), 4))

assert log_loss(y, cal_p) < log_loss(y, raw_p)

### Step 3 - Recompute ECE after calibration

The calibrated probabilities should be closer to observed frequencies.

In [ ]:
bin_id_cal = np.digitize(cal_p, bins) - 1
rows_cal = []
for b in range(len(bins) - 1):
    mask = bin_id_cal == b
    if mask.sum() > 0:
        rows_cal.append((b, mask.sum(), cal_p[mask].mean(), y[mask].mean()))

rel_cal = pd.DataFrame(rows_cal, columns=["bin", "n", "conf", "acc"])
rel_cal["gap"] = (rel_cal["acc"] - rel_cal["conf"]).abs()
ece_cal = ((rel_cal["n"] / n) * rel_cal["gap"]).sum()

print("cal ECE:", round(ece_cal, 4))

assert ece_cal < ece_raw

## Visualize reliability

Perfect calibration sits on the diagonal. Points below the line mean overprediction.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot([0, 1], [0, 1], color="black", linewidth=1)
ax.scatter(reliability["conf"], reliability["acc"], label="raw")
ax.scatter(rel_cal["conf"], rel_cal["acc"], label="Platt")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed frequency")
ax.set_title("reliability diagram")
ax.legend()
plt.show()

## Practice

Try each in the empty cell below it.

1. Change the number of bins to 12 and compare ECE stability.
2. Simulate underconfidence by setting `true_p = 1.3 * raw_p` clipped to 1.
3. Print the learned Platt coefficient and intercept.

In [ ]:
# Your turn:
